In [ ]:
import cv2
import os
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.svm import SVC
import joblib

# --- 1. CONFIGURATION ---
# LBPH Config (For the OpenCV model)
RADIUS = 1
NEIGHBORS = 8
GRID_X = 8
GRID_Y = 8

# --- 2. HELPER FUNCTIONS ---
def rotate_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    return rotated

def extract_hog_features(image):
    """
    Extracts Histogram of Oriented Gradients (HOG) features.
    
    Parameters:
    - orientations: Number of orientation bins (usually 9).
    - pixels_per_cell: Size of a cell (8x8 is standard for faces).
    - cells_per_block: Normalization block size (2x2 is standard).
    """
    # Using visualization=False returns just the feature vector
    features = hog(image, 
                   orientations=9, 
                   pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), 
                   block_norm='L2-Hys', 
                   visualize=False)
    return features

# --- 3. DATA LOADING & AUGMENTATION ---
def load_and_augment_data(path):
    image_paths = [os.path.join(path, f) for f in os.listdir(path)]
    
    # Lists for LBPH (Raw Images - OpenCV handles LBP internally)
    lbph_faces = []
    lbph_ids = []
    
    # Lists for SVM (HOG Feature Vectors)
    svm_features = []
    svm_labels = []
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    print(f"Processing {len(image_paths)} source images...")
    
    for img_path in image_paths:
        try:
            img = Image.open(img_path).convert('L')
            img_np = np.array(img, 'uint8')
            
            # CAUTION: Ensure filenames are "User.ID.jpg"
            user_id = int(os.path.split(img_path)[-1].split(".")[1])
            
            # Base Preprocessing
            # Note: HOG is sensitive to image size. 200x200 is used here.
            face_resized = cv2.resize(img_np, (200, 200), interpolation=cv2.INTER_CUBIC)
            face_smooth = cv2.bilateralFilter(face_resized, 5, 75, 75)
            enhanced = clahe.apply(face_smooth)
            
            # --- AUGMENTATION STACK ---
            aug_imgs = [enhanced]
            aug_imgs.append(cv2.flip(enhanced, 1))                        # Flip
            aug_imgs.append(rotate_image(enhanced, -10))                  # Rotate Left
            aug_imgs.append(rotate_image(enhanced, 10))                   # Rotate Right
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=-40)) # Darker
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=40))  # Brighter
            
            # Add to datasets
            for face in aug_imgs:
                # 1. For LBPH: Add the raw image (OpenCV does LBP internally)
                lbph_faces.append(face)
                lbph_ids.append(user_id)
                
                # 2. For SVM: Extract HOG features and add
                hog_feat = extract_hog_features(face)
                svm_features.append(hog_feat)
                svm_labels.append(user_id)
                
        except Exception as e:
            print(f"Skipping {img_path}: {e}")
            
    return lbph_faces, lbph_ids, svm_features, svm_labels

# --- 4. EXECUTION ---
# Ensure this path exists on your machine
data_path = r'C:\Users\hp\Desktop\Attendance-System-Using-Face-Recognition\Dataset\training\Cleaned_Training'

faces, ids, features, labels = load_and_augment_data(data_path)

if len(faces) > 0:
    print(f"Total Augmented Samples: {len(faces)}")
    
    # --- TRAIN LBPH (OpenCV Native) ---
    print("Training LBPH Model (OpenCV)...")
    lbph = cv2.face.LBPHFaceRecognizer_create(radius=RADIUS, neighbors=NEIGHBORS, grid_x=GRID_X, grid_y=GRID_Y)
    lbph.train(faces, np.array(ids))
    lbph.save('trainer.yml')
    print("Saved 'trainer.yml'")
    
    # --- TRAIN SVM (HOG) ---
    print("Training SVM Model (HOG)...")
    # Note: HOG feature vectors are larger than LBP, so training might take slightly longer.
    svm = SVC(kernel='linear', C=10.0, gamma='scale', probability=True, random_state=42)
    svm.fit(features, labels)
    joblib.dump(svm, 'svm_face_model.pkl')
    print("Saved 'svm_face_model.pkl'")
    
else:
    print("No data found.")a

Processing 139 source images...
Total Augmented Samples: 834
Training LBPH Model (OpenCV)...
Saved 'trainer.yml'
Training SVM Model (HOG)...
Saved 'svm_face_model.pkl'
